In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


In [ ]:
load_dotenv()

model = ChatOpenAI()

In [27]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [28]:
def generate_joke(state: JokeState):
    prompt= f"Generate joke on topic: {state.get("topic")}"
    response = model.invoke(prompt).content
    return {"joke": response}

def generate_explanation(state: JokeState):
    prompt= f"Generate an explanation of following joke: {state.get("joke")}"
    response = model.invoke(prompt).content
    return {"explanation": response}

In [29]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explanation",generate_explanation)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","generate_explanation")
graph.add_edge("generate_explanation",END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [30]:
initial_state={"topic": "pizza"}
config1 = {"configurable":{"thread_id":"1"}}
final_state=workflow.invoke(initial_state,config=config1)
print(final_state)

{'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why was the pizza in a bad mood?" This primes the listener to expect a reason related to the pizza\'s emotional state or its situation.\n\nThe punchline is: "Because it was feeling crusty!"\n\nThe word "crusty" has a double meaning here:\n\n1. In baking, the crust is the outer layer of a pizza, which is crispy and crunchy.\n2. In everyday language, "crusty" can also mean being grumpy, irritable, or in a bad mood.\n\nThe joke relies on this wordplay to create a pun. The listener expects a reason for the pizza\'s bad mood, but instead, the joke uses the word "crusty" in a clever and unexpected way, connecting the pizza\'s physical characteristic (having a crust) to its emotional state (being grump

In [31]:
workflow.get_state(config1)
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why was the pizza in a bad mood?" This primes the listener to expect a reason related to the pizza\'s emotional state or its situation.\n\nThe punchline is: "Because it was feeling crusty!"\n\nThe word "crusty" has a double meaning here:\n\n1. In baking, the crust is the outer layer of a pizza, which is crispy and crunchy.\n2. In everyday language, "crusty" can also mean being grumpy, irritable, or in a bad mood.\n\nThe joke relies on this wordplay to create a pun. The listener expects a reason for the pizza\'s bad mood, but instead, the joke uses the word "crusty" in a clever and unexpected way, connecting the pizza\'s physical characteristic (having a crust) to its emotio

In [32]:
initial_state={"topic": "pasta"}
config2 = {"configurable":{"thread_id":2}}
final_state=workflow.invoke(initial_state,config=config2)
print(final_state)

{'topic': 'pasta', 'joke': "Here's one:\n\nWhy did the spaghetti refuse to get dressed?\n\nBecause it was feeling a little saucy!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why did the spaghetti refuse to get dressed?" This is a classic example of a "character doing something unexpected" - in this case, spaghetti, a type of food, is being personified and refusing to get dressed.\n\nThe punchline is: "Because it was feeling a little saucy!" This is where the wordplay happens. "Saucy" has a double meaning here:\n\n1. In cooking, "saucy" refers to having a sauce, which is a common accompaniment to spaghetti.\n2. In everyday language, "feeling saucy" is an idiomatic expression that means feeling playful, cheeky, or flirtatious.\n\nThe joke relies on the listener being familiar with both meanings of "saucy". The humor comes from the unexpected twist 

In [33]:
workflow.get_state(config2)
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Here's one:\n\nWhy did the spaghetti refuse to get dressed?\n\nBecause it was feeling a little saucy!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why did the spaghetti refuse to get dressed?" This is a classic example of a "character doing something unexpected" - in this case, spaghetti, a type of food, is being personified and refusing to get dressed.\n\nThe punchline is: "Because it was feeling a little saucy!" This is where the wordplay happens. "Saucy" has a double meaning here:\n\n1. In cooking, "saucy" refers to having a sauce, which is a common accompaniment to spaghetti.\n2. In everyday language, "feeling saucy" is an idiomatic expression that means feeling playful, cheeky, or flirtatious.\n\nThe joke relies on the listener being familiar with both meanings of "saucy". The humor comes from

TIME TRAVEL

In [34]:
workflow.get_state({"configurable":{"thread_id":"1","checkpoint_id":"1f128570-f8b8-6c21-8000-6a61e18518ae"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f128570-f8b8-6c21-8000-6a61e18518ae'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-25T14:29:41.326530+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128570-f8b4-66fc-bfff-e1e5956c707e'}}, tasks=(PregelTask(id='72e9e8f5-6110-0c1b-14fe-163e8e75f986', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!"}),), interrupts=())

In [35]:
workflow.invoke(None,config={"configurable":{"thread_id":"1","checkpoint_id":"1f128570-f8b8-6c21-8000-6a61e18518ae"}})

{'topic': 'pizza',
 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!",
 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why was the pizza in a bad mood?" This primes the listener to expect a reason related to the pizza\'s emotional state or its situation.\n\nThe punchline is: "Because it was feeling crusty!"\n\nThe word "crusty" has a double meaning here:\n\n1. In the context of pizza, "crusty" refers to the crust of a pizza, which is the outer layer of the bread.\n2. In a more general sense, "crusty" can also mean being grumpy or irritable, often due to being in a bad mood.\n\nThe joke relies on this wordplay to create a pun. The listener is expecting a reason why the pizza is in a bad mood, and the answer "it was feeling crusty" is a clever connection between the pizza\'s physical characteristic (having a cru

In [36]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why was the pizza in a bad mood?" This primes the listener to expect a reason related to the pizza\'s emotional state or its situation.\n\nThe punchline is: "Because it was feeling crusty!"\n\nThe word "crusty" has a double meaning here:\n\n1. In the context of pizza, "crusty" refers to the crust of a pizza, which is the outer layer of the bread.\n2. In a more general sense, "crusty" can also mean being grumpy or irritable, often due to being in a bad mood.\n\nThe joke relies on this wordplay to create a pun. The listener is expecting a reason why the pizza is in a bad mood, and the answer "it was feeling crusty" is a clever connection between the pizza\'s physical character

In [38]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why was the pizza in a bad mood?" This primes the listener to expect a reason related to the pizza\'s emotional state or its situation.\n\nThe punchline is: "Because it was feeling crusty!"\n\nThe word "crusty" has a double meaning here:\n\n1. In the context of pizza, "crusty" refers to the crust of a pizza, which is the outer layer of the bread.\n2. In a more general sense, "crusty" can also mean being grumpy or irritable, often due to being in a bad mood.\n\nThe joke relies on this wordplay to create a pun. The listener is expecting a reason why the pizza is in a bad mood, and the answer "it was feeling crusty" is a clever connection between the pizza\'s physical characte

In [41]:
workflow.update_state({"configurable":{"thread_id":"1","checkpoint_id":"1f128570-f8b8-6c21-8000-6a61e18518ae","checkpoint_ns":""}},{"topic":"samosa"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f128583-69f1-6c25-8001-d162b0c154b1'}}

In [43]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128583-69f1-6c25-8001-d162b0c154b1'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-25T14:37:56.382601+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128570-f8b8-6c21-8000-6a61e18518ae'}}, tasks=(PregelTask(id='9eab0cfd-7949-ff74-ed07-00720171a7d9', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling crusty!\n\nHope that made you laugh!", 'explanation': 'The joke!\n\nThis joke is a play on words, which is a common technique used in humor. Here\'s how it works:\n\nThe setup for the joke is: "Why was the pizza in a bad mood?" This primes the listener to 

In [44]:
workflow.invoke(None,config={"configurable":{"thread_id":"1","checkpoint_id":"1f128583-69f1-6c25-8001-d162b0c154b1"}})

{'topic': 'samosa',
 'joke': 'Here\'s one:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling a little "crusty" on the outside and worried it was "folding" under the pressure!\n\nHope that made you laugh!',
 'explanation': 'A clever joke! Let\'s break it down:\n\nThe joke is a play on words, using the characteristics of a samosa (a type of fried or baked pastry) to create a pun.\n\nThe setup is straightforward: a samosa goes to therapy, implying that it\'s experiencing some emotional distress.\n\nThe punchline has two parts:\n\n1. "Crusty on the outside": A samosa is a pastry with a crispy, crunchy exterior, so it\'s a literal description of its physical state. However, "crusty" can also imply being grumpy, irritable, or emotionally hardened. The joke relies on this dual meaning, suggesting that the samosa is not only physically crusty but also feeling emotionally tough or defensive on the outside.\n2. "Folding under the pressure": This phrase is a common idiomatic expressi

In [45]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Here\'s one:\n\nWhy did the samosa go to therapy?\n\nBecause it was feeling a little "crusty" on the outside and worried it was "folding" under the pressure!\n\nHope that made you laugh!', 'explanation': 'A clever joke! Let\'s break it down:\n\nThe joke is a play on words, using the characteristics of a samosa (a type of fried or baked pastry) to create a pun.\n\nThe setup is straightforward: a samosa goes to therapy, implying that it\'s experiencing some emotional distress.\n\nThe punchline has two parts:\n\n1. "Crusty on the outside": A samosa is a pastry with a crispy, crunchy exterior, so it\'s a literal description of its physical state. However, "crusty" can also imply being grumpy, irritable, or emotionally hardened. The joke relies on this dual meaning, suggesting that the samosa is not only physically crusty but also feeling emotionally tough or defensive on the outside.\n2. "Folding under the pressure": This phrase is a commo